In [1]:
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np

## Load Bible

In [2]:
with open("SEPTUAGINT.xml") as f:
    sept_soup = BeautifulSoup(f.read(), "xml")

with open("TISCHENDORF.xml") as file:
    tisch_soup = BeautifulSoup(file, "xml")

rm_punct_tbl = str.maketrans("", "", " .·,:;!()«»-·")
sept = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str", "G").removeprefix("G"),
            e.find_parent("VERS")["vnumber"],
            e.find_parent("CHAPTER")["cnumber"],
            e.find_parent("BIBLEBOOK")["bnumber"],
            e["rmac"],
        )
        for e in sept_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)
tisch = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str"),
            e.find_parent("VERS")["vnumber"],
            e.find_parent("CHAPTER")["cnumber"],
            e.find_parent("BIBLEBOOK")["bnumber"],
            e["rmac"],
        )
        for e in tisch_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)

In [3]:
sept["verse"] = sept["verse"].astype(int)
sept["chapter"] = sept["chapter"].astype(int)
sept["book"] = sept["book"].astype(int)
# include code to parse rmac?

In [4]:
tisch["verse"] = tisch["verse"].astype(int)
tisch["chapter"] = tisch["chapter"].astype(int)
tisch["book"] = tisch["book"].astype(int)

## Load Dictionary

In [32]:
import re

with open("./strongs-dictionary.xhtml") as f:
    strong_soup = BeautifulSoup(f.read(), "xml")

[_, nt] = strong_soup.find_all("section")
strongs = pd.DataFrame(
    [
        (
            str(e["value"]),
            str(e.find("i").text),
            str(
                (
                    kjv_def if (kjv_def := e.find(class_="kjv_def")) is not None else e
                ).text
            ),
        )
        for e in nt.find_all("li")
    ],
    columns=["str", "original", "def"],
).set_index("str")

## Reconciling missing strongs

In [5]:
# words without strongs
nameless = pd.concat(
    [tisch[~tisch["str"].astype(bool)], sept[~sept["str"].astype(bool)]]
)

# unique
no_str_words = pd.Series(nameless["text"].unique())


new_strongs = pd.Series(
    ["C" + str(number) for number in range(0, len(no_str_words))], index=no_str_words
)

In [6]:
# if null or "" sets to False, anything else -> True.
tisch_needs_new = ~tisch["str"].astype(bool)
tisch.loc[tisch_needs_new, "str"] = new_strongs.loc[
    tisch.loc[tisch_needs_new, "text"]
].values

sept_needs_new = ~sept["str"].astype(bool)
sept.loc[sept_needs_new, "str"] = new_strongs.loc[
    sept.loc[sept_needs_new, "text"]
].values

## Pickling

In [34]:
import pickle

with open("./pickles/tisch.pickle", "wb") as f:
    pickle.dump(tisch, f)
with open("./pickles/sept.pickle", "wb") as f:
    pickle.dump(sept, f)
with open("./pickles/strongs.pickle", "wb") as f:
    pickle.dump(strongs, f)